# Extractive Summarization — BERT

Metode BERT: memilih kalimat berdasarkan MAKNA (embedding semantik), bukan kata. Kalimat yang maknanya paling mewakili tema artikel dipilih. Tetap extractive (memilih kalimat asli). Input: `data_final_preprocessing.csv` (sudah dipreprocessing).

**Struktur:** Load → Stopword → Fungsi → ROUGE → BERTScore → Contoh → Simpan.

---
## Setup

In [1]:
!pip install sentence-transformers rouge-score bert-score -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 97.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26

In [2]:
import re
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import sent_tokenize
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from rouge_score import rouge_scorer
from bert_score import score as bertscore
from tqdm import tqdm
tqdm.pandas()

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

Device: cuda


---
## 1. Load Dataset

In [3]:
df = pd.read_csv('/kaggle/input/datasets/nazhifberlian/nlp-wikipedia-summarization/data_final_preprocessing.csv',
                 encoding='utf-8-sig')

print(f'Dataset dimuat: {len(df)} baris, {len(df.columns)} kolom')
print(f'Kolom: {list(df.columns)}')
print(f'\nDistribusi kategori:')
print(df['category'].value_counts().to_string())

Dataset dimuat: 12644 baris, 9 kolom
Kolom: ['global_id', 'id', 'title', 'category', 'article_text', 'body_word_count', 'lead_paragraph', 'lead_word_count', 'sentences']

Distribusi kategori:
category
sejarah     1976
arts        1962
artis       1947
kuliner     1947
tech        1716
biografi    1679
sains       1417


---
## 2. Muat Model Embedding
* BERT memilih berdasarkan makna, jadi tidak memakai stopword removal seperti TF-IDF. Di tahap ini model embedding dimuat.
* Model: MiniLM multilingual (mendukung bahasa Indonesia).

In [4]:
bert_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2', device=DEVICE)
print(f'Model BERT dimuat: {bert_model.get_sentence_embedding_dimension()} dimensi')

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model BERT dimuat: 384 dimensi


/tmp/ipykernel_23/3492972996.py:2: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f'Model BERT dimuat: {bert_model.get_sentence_embedding_dimension()} dimensi')


---
## 3. Fungsi Summarization

In [5]:
def split_sentences(text):
    sentences = sent_tokenize(str(text))
    return [s.strip() for s in sentences if len(s.split()) >= 4]

def adaptive_n(text):
    n = len(str(text).split())
    if n < 500:  return 2
    if n < 1500: return 3
    return 4

def summarize_bert(text, n_sentences=3):
    sentences = split_sentences(text)
    if len(sentences) <= n_sentences:
        return ' '.join(sentences)
    embeddings = bert_model.encode(sentences, show_progress_bar=False)
    doc_embedding = embeddings.mean(axis=0, keepdims=True)
    scores = cosine_similarity(embeddings, doc_embedding).flatten()
    top_idx = sorted(scores.argsort()[-n_sentences:])
    return ' '.join(sentences[i] for i in top_idx)

print('Fungsi BERT siap.')

Fungsi BERT siap.


### Generate ringkasan untuk seluruh dataset

In [6]:
print('Generating ringkasan BERT...')
start = time.time()

df['summary_bert'] = df['article_text'].progress_apply(
    lambda x: summarize_bert(x, n_sentences=adaptive_n(x)))

time_bert = time.time() - start
print(f'Selesai dalam {time_bert:.1f} detik ({time_bert/len(df):.3f} detik/artikel)')

Generating ringkasan BERT...


100%|██████████| 12644/12644 [06:30<00:00, 32.41it/s]

Selesai dalam 390.2 detik (0.031 detik/artikel)


---
## 4. Evaluasi ROUGE

In [7]:
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)

r1, r2, rl = [], [], []
for pred, ref in zip(df['summary_bert'], df['lead_paragraph']):
    s = scorer.score(str(ref), str(pred))
    r1.append(s['rouge1'].fmeasure)
    r2.append(s['rouge2'].fmeasure)
    rl.append(s['rougeL'].fmeasure)

rouge_bert = (np.mean(r1), np.mean(r2), np.mean(rl))

print('Evaluasi ROUGE — BERT')
print(f'{"Metrik":<10} | {"Skor":>8}')
print('-' * 22)
print(f'{"ROUGE-1":<10} | {rouge_bert[0]:>8.4f}')
print(f'{"ROUGE-2":<10} | {rouge_bert[1]:>8.4f}')
print(f'{"ROUGE-L":<10} | {rouge_bert[2]:>8.4f}')

Evaluasi ROUGE — BERT
Metrik     |     Skor
----------------------
ROUGE-1    |   0.1927
ROUGE-2    |   0.0445
ROUGE-L    |   0.1206


### ROUGE-L per Kategori

In [8]:
rows = []
for kat in sorted(df['category'].unique()):
    sub = df[df['category'] == kat]
    scores = [scorer.score(str(ref), str(pred))['rougeL'].fmeasure
              for pred, ref in zip(sub['summary_bert'], sub['lead_paragraph'])]
    rows.append({'kategori': kat, 'jumlah': len(sub), 'rougeL': np.mean(scores)})

hasil_kategori = pd.DataFrame(rows).sort_values('rougeL', ascending=False)
print('ROUGE-L per kategori (BERT):')
print(hasil_kategori.round(4).to_string(index=False))

ROUGE-L per kategori (BERT):
kategori  jumlah  rougeL
    tech    1716  0.1298
 sejarah    1976  0.1286
 kuliner    1947  0.1267
   sains    1417  0.1185
biografi    1679  0.1163
   artis    1947  0.1122
    arts    1962  0.1118


---
## 5. Evaluasi BERTScore
CATATAN BIAS: BERTScore memakai model multilingual default (lang='id'), berbeda  
dari MiniLM yang dipakai untuk memilih kalimat di atas untuk menghindari bias evaluasi.

In [9]:
SAMPLE_SIZE_BS = 500
sample_bs = df.sample(n=min(SAMPLE_SIZE_BS, len(df)), random_state=42).reset_index(drop=True)

cands = sample_bs['summary_bert'].astype(str).tolist()
refs  = sample_bs['lead_paragraph'].astype(str).tolist()

P, R, F1 = bertscore(cands, refs, lang='id', verbose=False)
bertscore_bert = (P.mean().item(), R.mean().item(), F1.mean().item())

print(f'Evaluasi BERTScore — BERT (sampel N={len(sample_bs)})')
print(f'{"Metrik":<12} | {"Skor":>8}')
print('-' * 24)
print(f'{"Precision":<12} | {bertscore_bert[0]:>8.4f}')
print(f'{"Recall":<12} | {bertscore_bert[1]:>8.4f}')
print(f'{"F1":<12} | {bertscore_bert[2]:>8.4f}')

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Evaluasi BERTScore — BERT (sampel N=500)
Metrik       |     Skor
------------------------
Precision    |   0.6856
Recall       |   0.6569
F1           |   0.6702


---
## 6. Contoh Ringkasan Hasil

In [10]:
N_CONTOH = 3
sample_idx = df.sample(N_CONTOH, random_state=42).index

for i, idx in enumerate(sample_idx):
    row = df.loc[idx]
    print('=' * 70)
    print(f'Artikel {i+1}: [{row["category"]}] {row["title"]}')
    print('-' * 20)
    print(f'Ground Truth (lead_paragraph):')
    print(f'{str(row["lead_paragraph"])[:200]}...')
    print('-' * 20)
    print(f'Ringkasan BERT:')
    print(f'{str(row["summary_bert"])[:200]}')
    print()

Artikel 1: [sains] Atom Bakhidrogen
--------------------
Ground Truth (lead_paragraph):
suatu ion lirhidrogen bahasa inggris hydrogenlike ioncode en is deprecated  disebut pula ion mirip hidrogen merupakan inti atom yang memiliki satu elektron dan karenanya isoelektronik dengan hidrogen....
--------------------
Ringkasan BERT:
dalam penyelesaian persamaan schrdinger, yang bersifat nonrelativistik, orbital atom bakhidrogen merupakan eigenfungsi dari operator momentum sudut satuelektron l dan komponen znya lz. suatu orbital a

Artikel 2: [kuliner] Jeruk Bergamot
--------------------
Ground Truth (lead_paragraph):
jeruk bergamot adalah buah jeruk wangi dengan warna kuning atau hijau mirip dengan jeruk nipis, tergantung pada kematangannya. penelitian genetika tentang asal muasal leluhur kultivar jeruk yang masih...
--------------------
Ringkasan BERT:
c. bergamia sering salah diidentifikasikan sebagai jeruk lain, c. hystrix jeruk purut, karena yang terakhir kadang disebut sebagai thai berg

---
## 7. Simpan Hasil

In [11]:
output_cols = ['global_id', 'title', 'category', 'summary_bert',
               'lead_paragraph', 'body_word_count']
hasil = df[[c for c in output_cols if c in df.columns]]

hasil.to_csv('hasil_summary_bert.csv', index=False, encoding='utf-8-sig')
print(f'Tersimpan: hasil_summary_bert.csv ({len(hasil)} baris)')
print(f'Kolom: {list(hasil.columns)}')

Tersimpan: hasil_summary_bert.csv (12644 baris)
Kolom: ['global_id', 'title', 'category', 'summary_bert', 'lead_paragraph', 'body_word_count']
